In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from emoatlas import EmoScores
import pickle
import networkx as nx

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
from scripts.bfmn_building import categorize_valence_multiple, pipeline_df_building, pipeline_single_participant, build_multiple_fmnt

Notes in BFMN building, the arguments of the function 'build_multiple_fmnt': 
- flower default is 'yes'
- language default is 'ita'
  
In the function 'pipeline_df_building':
- filter default is 'no'

If not inserted as argument of the function these will be taken as default values, if you want to change it call the argument with:
- filter = 'yes' --> if you want to filter the idiosyncratic associations (the ones made by only one participant)
- flower = 'no' --> if you don't want to plot emotional flowers
- language = 'eng' --> if you want EmoAtlas to work in English

In [ ]:
path_rassereno = '/Users/emma/projects/cognoscolab/rassereno'

## Dataset Scuola Fisica

In [ ]:
liceo_luigi = pd.read_csv('/Users/emma/projects/cognoscolab/rassereno/out/formatting/liceo_2025/associations_valence_liceo_2025.csv')

liceo_luigi_001 = liceo_luigi[liceo_luigi['participant_id'] == 'liceo_063'].copy()
liceo_luigi_002 = liceo_luigi[liceo_luigi['participant_id'] == 'liceo_064'].copy()

final_df_1, G_1, features_fmnt_1, valences_1 = pipeline_single_participant(liceo_luigi_001)
final_df_2, G_2, features_fmnt_2, valences_2 = pipeline_single_participant(liceo_luigi_002)

In [ ]:
emo = EmoScores(language="italian", spacy_model='it_core_news_lg')

In [ ]:
fmn = emo.nxgraph_to_formamentis(G_1)
for keyword in ['fisica', 'scienza', 'matematica', 'ansia', 'esame']:

    # plot bfmn
    fig, ax = plt.subplots(figsize=(9,9))
    ax.axis('off')       
    fmnt_keyword = emo.extract_word_from_formamentis(fmn, keyword)
    emo.draw_formamentis(
        fmn=fmnt_keyword,
        layout="edge_bundling",
        custom_valences=valences_1, 
        ax=ax,
    )
    fig.savefig(f'{path_rassereno}/out/analysis/bfmn/luigi_df/liceo_063_{keyword}.png', bbox_inches="tight", pad_inches=0.1)
fig, ax = plt.subplots(figsize=(14,14))
ax.axis('off')       
emo.draw_formamentis(
    fmn=fmn,
    layout="edge_bundling",
    custom_valences=valences_1, 
    ax=ax,
)
fig.savefig(f'{path_rassereno}/out/analysis/bfmn/luigi_df/liceo_063_overall.png', bbox_inches="tight", pad_inches=0.1)

In [ ]:
fmn = emo.nxgraph_to_formamentis(G_2)
for keyword in ['fisica', 'scienza', 'matematica', 'ansia', 'esame']:

    # plot bfmn
    fig, ax = plt.subplots(figsize=(9,9))
    ax.axis('off')       
    fmnt_keyword = emo.extract_word_from_formamentis(fmn, keyword)
    emo.draw_formamentis(
        fmn=fmnt_keyword,
        layout="edge_bundling",
        custom_valences=valences_2, 
        ax=ax,
    )
    fig.savefig(f'/Users/emma/projects/cognoscolab/rassereno/out/analysis/bfmn/luigi_df/liceo_064_{keyword}.png', bbox_inches="tight", pad_inches=0.1)
fig, ax = plt.subplots(figsize=(14,14))
ax.axis('off')       
emo.draw_formamentis(
    fmn=fmn,
    layout="edge_bundling",
    custom_valences=valences_2, 
    ax=ax,
)
fig.savefig(f'/Users/emma/projects/cognoscolab/rassereno/out/analysis/bfmn/luigi_df/liceo_064_overall.png', bbox_inches="tight", pad_inches=0.1)

## Files reading

In [ ]:
students_2019 = pd.read_csv(f'{path_rassereno}/out/formatting/students_2019/associations_valence_students_2019.csv')
experts_2019 = pd.read_csv(f'{path_rassereno}/out/formatting/experts_2019/associations_valence_experts_2019.csv')

physics_2025 = pd.read_csv(f'{path_rassereno}/out/formatting/physics_2025/associations_valence_fisica.csv')
physics_gpt_oss = pd.read_csv(f'{path_rassereno}/out/formatting/gpt_oss/associations_valence_physics.csv')

In [ ]:
psychology_high_anx = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/all_psychology_students/high_anx.csv')
psychology_low_anx = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/all_psychology_students/low_anx.csv') 

In [ ]:
gpt_oss_psychology_high_anx = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/psychology_gpt_oss/high_anx.csv')
gpt_oss_psychology_low_anx = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/psychology_gpt_oss/low_anx.csv')

In [ ]:
high_school_2025_high_anx = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/high_school_2025/high_anx.csv')
high_school_2025_low_anx = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/high_school_2025/low_anx.csv')

In [ ]:
gpt_oss_high_school_high_anx = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/high_school_gpt_oss/high_anx.csv')
gpt_oss_high_school_low_anx = pd.read_csv(f'{path_rassereno}/out/splitted_dfs/high_school_gpt_oss/low_anx.csv')

## Dataframe preparation

### Experts

In [ ]:
df_experts_2019 = categorize_valence_multiple(experts_2019)

### High school 2019

In [ ]:
df_high_school_2019 = categorize_valence_multiple(students_2019)

In [ ]:
col_interest = ['cue_word', 'word']
for col in col_interest:
    if col in df_high_school_2019.columns:
        df_high_school_2019[col] = (
            df_high_school_2019[col]
            .astype(str)
            .str.lower()
            .str.replace('à', 'a')
        )

### Physics 2025

In [ ]:
df_physics_students_2025 = pipeline_df_building(physics_2025)

### GPT-oss Physics

In [ ]:
df_gpt_oss_physics_2025 = pipeline_df_building(physics_gpt_oss)

### High school 2025

In [ ]:
df_high_school_2025_high_anx = pipeline_df_building(high_school_2025_high_anx)

In [ ]:
df_high_school_2025_low_anx = pipeline_df_building(high_school_2025_low_anx)

### GPT-oss high school

In [ ]:
df_gpt_oss_high_school_high_anx = pipeline_df_building(gpt_oss_high_school_high_anx)

In [ ]:
df_gpt_oss_high_school_low_anx = pipeline_df_building(gpt_oss_high_school_low_anx)

### Psychology 2025

In [ ]:
df_psychology_high_anx_filtered = pipeline_df_building(psychology_high_anx, filter='yes')

In [ ]:
df_psychology_high_anx_unfiltered = pipeline_df_building(psychology_high_anx, filter='no')

In [ ]:
df_psychology_low_anx_filtered = pipeline_df_building(psychology_low_anx, filter='yes')

In [ ]:
df_psychology_low_anx_unfiltered = pipeline_df_building(psychology_low_anx, filter='no')

### GPT-oss psychology

In [ ]:
df_gpt_oss_psychology_high_anx = pipeline_df_building(gpt_oss_psychology_high_anx)

In [ ]:
df_gpt_oss_psychology_low_anx = pipeline_df_building(gpt_oss_psychology_low_anx)

## BFMN building

In [ ]:
keywords_rassereno = ['matematica', 'statistica', 'fisica', 'scienza', 'informatica', 'ricerca', 'esame', 'voto', 'professore', 'insegnante', 'scienziato', 'scienziata', 'scuola', 'universita', 'ansia', 'benessere']

keywords_rassereno_eng = ['mathematics', 'statistics', 'physics', 'science', 'computer', 'research', 'exam','grade', 'professor', 'teacher', 'scientist', 'school', 'university', 'anxiety', 'wellbeing']

### Experts 2019

In [ ]:
fmnts_experts, features_keyword_fmns_experts, features_fmn_tot_experts = build_multiple_fmnt(df_experts_2019, keywords_rassereno_eng, path_rassereno, 'experts_2019', language="eng")

### High school students 2019

In [ ]:
fmnts_liceo_2019, features_keyword_fmns_liceo_2019, features_fmn_tot_liceo_2019 = build_multiple_fmnt(df_high_school_2019, keywords_rassereno, path_rassereno, 'liceo_2019')

### High school 2025

Students

In [ ]:
fmnts_liceo_2025_high_anx, features_keyword_fmns_liceo_2025_high_anx, features_fmn_tot_liceo_2025_high_anx = build_multiple_fmnt(df_high_school_2025_high_anx, keywords_rassereno, path_rassereno, 'liceo_2025/high_anxiety')

In [ ]:
fmnts_liceo_2025_low_anx, features_keyword_fmns_liceo_2025_low_anx, features_fmn_tot_liceo_2025_low_anx = build_multiple_fmnt(df_high_school_2025_low_anx, keywords_rassereno, path_rassereno, 'liceo_2025/low_anxiety')

GPT-oss

In [ ]:
fmnts_gpt_liceo_high_anx, features_keyword_fmns_gpt_liceo_high_anx, features_fmn_tot_gpt_liceo_high_anx = build_multiple_fmnt(df_gpt_oss_high_school_high_anx, keywords_rassereno, path_rassereno, 'gpt_oss_high_school/high_anxiety')

In [ ]:
fmnts_gpt_liceo_low_anx, features_keyword_fmns_gpt_liceo_low_anx, features_fmn_tot_gpt_liceo_low_anx = build_multiple_fmnt(df_gpt_oss_high_school_low_anx, keywords_rassereno, path_rassereno, 'gpt_oss_high_school/low_anxiety')

### Physics

Students

In [ ]:
fmnts_physics, features_keyword_fmns_physics, features_fmn_tot_physics = build_multiple_fmnt(df_physics_students_2025, keywords_rassereno, path_rassereno, 'physics_students_2025')

GPT-oss

In [ ]:
fmnts_gpt_physics, features_keyword_fmns_gpt_physics, features_fmn_tot_gpt_physics = build_multiple_fmnt(df_gpt_oss_physics_2025, keywords_rassereno, path_rassereno, 'gpt_oss_physics')

### High anxiety psychology students

In [ ]:
fmnts_psy_high_anx, features_keyword_fmns_psy_high_anx, features_fmn_tot_psy_high_anx = build_multiple_fmnt(df_psychology_high_anx_filtered, keywords_rassereno, path_rassereno, 'psychology_students_filtered/high_anxiety', flower='no')

In [ ]:
fmnt_ps_high_anx_unfiltered, features_keyword_fmns_psy_high_anx_unfiltered, features_fmn_tot_psy_high_anx_unfiltered = build_multiple_fmnt(df_psychology_high_anx_unfiltered, keywords_rassereno, path_rassereno, 'psychology_students_unfiltered/high_anxiety', flower='yes')

### Low anxiety psychology students

In [ ]:
fmnts_psy_low_anx, features_keyword_fmns_psy_low_anx, features_fmn_tot_psy_low_anx = build_multiple_fmnt(df_psychology_low_anx_filtered, keywords_rassereno, path_rassereno, 'psychology_students_filtered/low_anxiety', flower='no')

In [ ]:
fmnts_psy_low_anx_unfiltered, features_keyword_fmns_psy_low_anx_unfiltered, features_fmn_tot_psy_low_anx_unfiltered = build_multiple_fmnt(df_psychology_low_anx_unfiltered, keywords_rassereno, path_rassereno, 'psychology_students_unfiltered/low_anxiety', flower='yes')

### Psychology students GPT-oss

In [ ]:
fmnts_gpt_psy_high_anx, features_keyword_fmns_gpt_psy_high_anx, features_fmn_tot_gpt_psy_high_anx = build_multiple_fmnt(df_gpt_oss_psychology_high_anx, keywords_rassereno, path_rassereno, 'gpt_oss_psychology/high_anxiety')

In [ ]:
fmnts_gpt_psy_low_anx, features_keyword_fmns_gpt_psy_low_anx, features_fmn_tot_gpt_psy_low_anx = build_multiple_fmnt(df_gpt_oss_psychology_low_anx, keywords_rassereno, path_rassereno, 'gpt_oss_psychology/low_anxiety')

## BFMN exports

In [ ]:
all_keyword_fmns_features = pd.concat([
    features_keyword_fmns_experts,
    features_keyword_fmns_liceo_2019,
    features_keyword_fmns_liceo_2025_high_anx,
    features_keyword_fmns_liceo_2025_low_anx,
    features_keyword_fmns_gpt_liceo_high_anx,
    features_keyword_fmns_gpt_liceo_low_anx,
    features_keyword_fmns_physics,
    features_keyword_fmns_gpt_physics,
    features_keyword_fmns_psy_high_anx,
    features_keyword_fmns_psy_low_anx,
    features_keyword_fmns_gpt_psy_high_anx,
    features_keyword_fmns_gpt_psy_low_anx
], ignore_index=True)

In [ ]:
all_keyword_fmns_features.to_csv(f'{path_rassereno}/out/analysis/features_keyword_fmns.csv', index=False)

In [ ]:
all_fmn_tot_features = pd.concat([
    features_fmn_tot_experts,
    features_fmn_tot_liceo_2019,
    features_fmn_tot_liceo_2025_high_anx,
    features_fmn_tot_liceo_2025_low_anx,
    features_fmn_tot_gpt_liceo_high_anx,
    features_fmn_tot_gpt_liceo_low_anx,
    features_fmn_tot_physics,
    features_fmn_tot_gpt_physics,
    features_fmn_tot_psy_high_anx,
    features_fmn_tot_psy_low_anx,
    features_fmn_tot_gpt_psy_high_anx,
    features_fmn_tot_gpt_psy_low_anx
], ignore_index=True)

In [ ]:
all_fmn_tot_features.to_csv(f'{path_rassereno}/out/analysis/features_tot_fmns.csv', index=False)

In [ ]:
all_bfmns = {
    'experts' : [fmnts_experts[idx].edges for idx in range(len(fmnts_experts))],
    'high_school_south' : [fmnts_liceo_2019[idx].edges for idx in range(len(fmnts_liceo_2019))],
    'high_school_high_anx' : [fmnts_liceo_2025_high_anx[idx].edges for idx in range(len(fmnts_liceo_2025_high_anx))], 
    'high_school_low_anx' :  [fmnts_liceo_2025_low_anx[idx].edges for idx in range(len(fmnts_liceo_2025_low_anx))],
    'gpt_oss_high_school_high_anx' :  [fmnts_gpt_liceo_high_anx[idx].edges for idx in range(len(fmnts_gpt_liceo_high_anx))],
    'gpt_oss_high_school_low_anx': [fmnts_gpt_liceo_low_anx[idx].edges for idx in range(len(fmnts_gpt_liceo_low_anx))],
    'physics' :  [fmnts_physics[idx].edges for idx in range(len(fmnts_physics))],
    'gpt_oss_physics' :  [fmnts_gpt_physics[idx].edges for idx in range(len(fmnts_gpt_physics))],
    'psychology_high_anx' :  [fmnts_psy_high_anx[idx].edges for idx in range(len(fmnts_psy_high_anx))],
    'psychology_low_anx' :  [fmnts_psy_low_anx[idx].edges for idx in range(len(fmnts_psy_low_anx))],
    'gpt_oss_psychology_high_anx' : [fmnts_gpt_psy_high_anx[idx].edges for idx in range(len(fmnts_gpt_psy_high_anx))],
    'gpt_oss_psychology_low_anx' :  [fmnts_gpt_psy_low_anx[idx].edges for idx in range(len(fmnts_gpt_psy_low_anx))]
}
with open(f"{path_rassereno}/out/analysis/all_bfmns.pkl", "wb") as f:
    pickle.dump(all_bfmns, f)